# 태양풍 속도 예측 — P3-renew (격자 제거 · 순수 코로나홀 마스크 피처)

**P3 와 모델이 같다.** 바뀐 것은 *코로나홀 마스크를 무엇으로 요약해서 GRU 에
넣는가* 뿐이다.

| | P3 | P3-renew |
|---|---|---|
| CH 피처 | 원반 3x5 격자의 셀별 면적 비율 **15개** | 마스크 전체의 스칼라 **1개**(기본) 또는 5개 |
| 탄도 정렬 | 중앙자오선 **셀** 시계열을 역산 인덱스에서 보간 | 전 원반 **면적** 시계열을 같은 방식으로 보간 |
| 마스크 규칙 | 193 AND 211 < 0.45 x 원반 중앙값 | 같음(`v1`) / `ch_mask_check` 수정본(`v2`) 선택 |
| 모델·손실·증강·Dataset·학습 루프 | — | **P3 와 동일** |

## 왜 격자를 버리나

격자 15개는 면적 1개를 **포함한다** (면적은 셀 면적들의 가중합이다). 그러므로
정보량으로는 renew 가 P3 의 부분집합이고, renew 가 이기는 경우는 하나뿐이다 —
**남은 14 자유도가 신호가 아니라 잡음이었을 때.** 격자를 넣어도 점수가 안 올랐다면
바로 그 상황을 의심하는 게 맞다. 이 노트북은 그 가설을 한 번의 학습으로 검정한다.

격자가 잡음이 되기 쉬운 이유도 분명하다. 셀 경계는 태양의 물리(자기 중립선·홀
경계)와 아무 상관 없는 **프레임 좌표 기준의 사각형**이고, 원반 검출이 몇 픽셀만
흔들려도 셀별 면적은 크게 요동친다. 반면 전 원반 면적은 그 흔들림에 훨씬 둔감하다.

> `CH_FEATURE_MODE = "area_shape"` 로 두면 면적에 **마스크의 1·2차 모멘트**
> (무게중심 동서·남북, 퍼짐)와 평균 깊이를 더한 5개를 쓴다. 격자처럼 공간을
> 미리 자르지 않고 마스크 자신이 위치를 말하게 하는 중간 단계다.

## 0. 설정

In [ ]:
from pathlib import Path
import gc
import json
import math
import os
import random
import shutil
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset

SEED = 777
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DATA_ROOT_CANDIDATES = [
    Path(os.getenv("SW_DATA_ROOT", "")) if os.getenv("SW_DATA_ROOT") else None,
    Path("public_dataset/competition_dataset_6h"),
    Path("/home/jovyan/public_dataset/competition_dataset_6h"),
    Path("public/public_dataset/competition_dataset_6h"),
    Path("/home/jovyan/public/public_dataset/competition_dataset_6h"),
    Path("dataset"),
    Path("/home/jovyan/dataset"),
]
DATA_ROOT = None
for candidate in DATA_ROOT_CANDIDATES:
    if candidate is not None and (candidate / "train/inputs.csv").exists():
        DATA_ROOT = candidate
        break
if DATA_ROOT is None:
    searched = "\n".join(f"  - {c}" for c in DATA_ROOT_CANDIDATES if c is not None)
    raise FileNotFoundError("데이터 경로를 찾지 못했습니다:\n" + searched)

WORK_DIR = Path("work")
CACHE_ROOT = WORK_DIR / "cache"
OUTPUT_DIR = WORK_DIR / "outputs_p3"
SUBMISSION_DIR = Path("submission")
for directory in (CACHE_ROOT, OUTPUT_DIR, SUBMISSION_DIR):
    directory.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 128
CHANNELS = ("193", "211")
BATCH_SIZE = 64
EPOCHS = 60
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-3
GRAD_CLIP = 1.0
SCHEDULER_PATIENCE = 3
EARLY_STOP_PATIENCE = 10
NUM_WORKERS = 4
LOSS_EPSILON = 1e-8
LOSS_SCALE = 100.0
LOSS_MODE = "metric"

# --- 브랜치 스위치 (ablation 용) ---------------------------------------
USE_CNN = False    # 3D CNN 영상 브랜치. P3 기본값은 끔 (과적합 주범)
USE_CH = True      # 코로나홀 격자 피처 브랜치
USE_BALLISTIC = True   # horizon별 탄도 정렬 피처

# --- 코로나홀 추출 (Collin 2025) ---------------------------------------
CH_GRID = (3, 5)          # (위도 구간, 경도 구간). 논문은 4x3 이 timeline RMSE 최적
CH_THRESHOLD_RATIO = 0.45  # 원반 중앙값 대비 이 비율보다 어두우면 코로나홀
DISK_MARGIN = 0.95         # 림 밝아짐(limb brightening) 회피용 반지름 축소
TRANSIT_SPEEDS = (350.0, 500.0, 700.0)   # 탄도 역산에 쓸 가정 속도 (km/s)
AU_KM = 1.496e8

DROPOUT = 0.4
HORIZON_EMBED = 8
AUGMENT = True
AUG_BRIGHTNESS = 0.10
AUG_SHIFT_PIXELS = 6
AUG_NOISE_STD = 0.02
AUG_ERASE_PROB = 0.3
AUG_CH_NOISE = 0.05        # CH 피처에 주는 곱셈 노이즈

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
PIN_MEMORY = DEVICE.type == "cuda"
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True
else:
    print("WARNING: CUDA Unavailable")

print("PyTorch:", torch.__version__, "| device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("data:", DATA_ROOT.resolve())
print(f"branches: CNN={USE_CNN} CH={USE_CH} BALLISTIC={USE_BALLISTIC}")

## 0-b. P3-renew 설정 — CH 피처 요약 방식

여기 있는 값만 바꾸면 된다. 아래 셀들은 P3 원본 그대로다.

- `CH_FEATURE_MODE` — `"area"`(기본, 스칼라 1개) / `"area_shape"`(5개)
- `CH_MASK_VERSION` — `"v1"`(P3 와 동일한 마스크) / `"v2"`([ch_mask_check.ipynb](ch_mask_check.ipynb) 수정본)

**첫 실행은 `v1` 로 둔다.** P3(val 64.203, Public 58.8028) 와의 차이가 오직
"격자 -> 면적" 하나가 되어야 무엇이 점수를 움직였는지 말할 수 있다. v2 는 그
다음 칸이다 — 마스크가 고장나 있다는 진단이 맞다면, 면적 하나로 줄인 renew 가
그 고장에 P3 보다 더 민감하다(격자에선 림 고리가 가장자리 셀에 몰려 모델이
무시할 수 있었지만, 면적에 섞이면 분리할 방법이 없다).

In [ ]:
# ===== P3-renew 노브 ==================================================
CH_FEATURE_MODE = "area"    # "area" = 면적 1개 | "area_shape" = 면적 + 마스크 모멘트 4개
CH_MASK_VERSION = "v1"      # "v1" = P3 와 동일 | "v2" = ch_mask_check.ipynb 의 수정 규칙

# --- v2 를 켤 때만 쓰인다. ch_mask_check.ipynb 셀 10 에서 고른 값을 옮겨 적는다 ---
V2_RATIO = 0.45             # 평탄화 후 "조용한 코로나 최빈값" 대비 비율
V2_CORE_FRACTION = 0.90     # 이 반지름 비율 안쪽만 코로나홀로 인정
V2_MIN_AREA = 0.0015        # 원반 넓이 대비 이보다 작은 조각은 버림 (scipy 필요)
V2_STACK_COUNT = 200        # 반경 프로파일에 쓸 train 프레임 수
RADIAL_BINS = 64            # 반경 방향 평탄화 구간 수

CH_FEATURE_NAMES = (["ch_area"] if CH_FEATURE_MODE == "area" else
                    ["ch_area", "ch_cx", "ch_cy", "ch_spread", "ch_depth"])
N_CH_FEATURES = len(CH_FEATURE_NAMES)

# 아래 두 이름은 P3 원본 셀(Dataset · 모델 · 탄도 정렬)이 그대로 참조한다.
# 셀을 고치지 않고 의미만 바꾼다 — "격자 셀 개수" -> "CH 피처 개수",
# "중앙자오선 셀 번호" -> "탄도 역산에 쓸 시계열의 열 번호".
N_CELLS = N_CH_FEATURES
CENTRAL_CELL = 0            # = ch_area. 전 원반 면적 시계열을 탄도 역산에 쓴다
CH_GRID = (1, 1)            # 격자 없음. 학습 셀의 CONFIG 기록 호환용으로만 남긴다

CH_RATIO = V2_RATIO if CH_MASK_VERSION == "v2" else CH_THRESHOLD_RATIO
EFFECTIVE_MARGIN = V2_CORE_FRACTION if CH_MASK_VERSION == "v2" else DISK_MARGIN
CH_CACHE_TAG = f"{CH_MASK_VERSION}_{CH_FEATURE_MODE}_r{CH_RATIO}_m{EFFECTIVE_MARGIN}_{IMAGE_SIZE}"

# P3 산출물을 덮어쓰지 않는다
OUTPUT_DIR = WORK_DIR / "outputs_p3_renew"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert CH_FEATURE_MODE in {"area", "area_shape"}, CH_FEATURE_MODE
assert CH_MASK_VERSION in {"v1", "v2"}, CH_MASK_VERSION
print(f"P3-renew — 마스크 {CH_MASK_VERSION} · 피처 {N_CH_FEATURES}개 {CH_FEATURE_NAMES}")
print(f"           임계비 {CH_RATIO} · 유효 반지름 x{EFFECTIVE_MARGIN}")
print(f"출력: {OUTPUT_DIR.resolve()}")

## 1. 데이터 로드 · 결측 처리

In [ ]:
IMAGE_COLUMNS = [f"image_{index:02d}" for index in range(20)]
WIND_COLUMNS = [f"wind_{index:02d}" for index in range(20)]
TARGET_COLUMNS = [f"target_{index:02d}" for index in range(12)]
HORIZONS = np.arange(1, 13) * 6

train_inputs = pd.read_csv(DATA_ROOT / "train/inputs.csv")
train_targets_frame = pd.read_csv(DATA_ROOT / "train/targets.csv")
val_inputs = pd.read_csv(DATA_ROOT / "validation/inputs.csv")
val_targets_frame = pd.read_csv(DATA_ROOT / "validation/targets.csv")
test_inputs = pd.read_csv(DATA_ROOT / "test/inputs.csv")
test_ids = pd.read_csv(DATA_ROOT / "test/test_ids.csv")

assert train_inputs.sample_id.tolist() == train_targets_frame.sample_id.tolist()
assert val_inputs.sample_id.tolist() == val_targets_frame.sample_id.tolist()
assert test_inputs.sample_id.tolist() == test_ids.sample_id.tolist()
assert set(train_inputs.sample_id).isdisjoint(val_inputs.sample_id)
assert set(train_inputs.sample_id).isdisjoint(test_inputs.sample_id)
assert set(val_inputs.sample_id).isdisjoint(test_inputs.sample_id)
assert not any(column.startswith("target_") for column in test_inputs.columns)


def forward_fill_rows(values):
    valid = np.isfinite(values)
    positions = np.where(valid, np.arange(values.shape[1])[None, :], 0)
    np.maximum.accumulate(positions, axis=1, out=positions)
    rows = np.arange(values.shape[0])[:, None]
    return np.where(valid.any(axis=1, keepdims=True), values[rows, positions], values)


def fill_wind(frame, fallback):
    values = frame[WIND_COLUMNS].to_numpy(np.float32)
    valid = np.isfinite(values).astype(np.float32)
    filled = forward_fill_rows(values)
    filled = forward_fill_rows(filled[:, ::-1])[:, ::-1]
    filled = np.where(np.isfinite(filled), filled, fallback)
    return np.ascontiguousarray(filled), np.ascontiguousarray(valid)


WIND_FALLBACK = float(np.nanmedian(train_inputs[WIND_COLUMNS].to_numpy(np.float32)))
train_wind, train_wind_valid = fill_wind(train_inputs, WIND_FALLBACK)
val_wind, val_wind_valid = fill_wind(val_inputs, WIND_FALLBACK)
test_wind, test_wind_valid = fill_wind(test_inputs, WIND_FALLBACK)

train_targets = train_targets_frame[TARGET_COLUMNS].to_numpy(np.float32)
val_targets = val_targets_frame[TARGET_COLUMNS].to_numpy(np.float32)
assert np.isfinite(train_targets).all() and np.isfinite(val_targets).all()

print("samples:", len(train_inputs), len(val_inputs), len(test_inputs))
print(f"train wind mean={train_wind.mean():.1f} target mean={train_targets.mean():.1f}")

## 2. 이미지 memory-map cache

In [ ]:
def prepare_image_memmap(split, inputs):
    image_root = DATA_ROOT / split
    cache_root = CACHE_ROOT / f"{IMAGE_SIZE}px"
    cache_root.mkdir(parents=True, exist_ok=True)
    array_path = cache_root / f"{split}_images.npy"
    metadata_path = cache_root / f"{split}_metadata.json"
    filenames = sorted(pd.unique(inputs[IMAGE_COLUMNS].to_numpy().ravel()).tolist())
    expected = {"image_size": IMAGE_SIZE, "channels": list(CHANNELS), "filenames": filenames}
    shape = (len(filenames), len(CHANNELS), IMAGE_SIZE, IMAGE_SIZE)

    valid = False
    if array_path.exists() and metadata_path.exists():
        try:
            metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
            cached = np.load(array_path, mmap_mode="r")
            valid = (metadata == expected and cached.shape == shape
                     and cached.dtype == np.uint8)
        except (OSError, ValueError, json.JSONDecodeError):
            valid = False

    if not valid:
        array_temp = array_path.with_name(array_path.name + f".partial.{os.getpid()}")
        metadata_temp = metadata_path.with_name(metadata_path.name + f".partial.{os.getpid()}")
        resized = np.lib.format.open_memmap(array_temp, mode="w+", dtype=np.uint8, shape=shape)
        resampling = Image.Resampling.BILINEAR
        for index, filename in enumerate(filenames):
            for channel_index, channel in enumerate(CHANNELS):
                with Image.open(image_root / channel / filename) as image:
                    resized[index, channel_index] = np.asarray(
                        image.convert("L").resize((IMAGE_SIZE, IMAGE_SIZE), resampling),
                        dtype=np.uint8)
            if (index + 1) % 2000 == 0 or index + 1 == len(filenames):
                print(f"{split} resize: {index + 1}/{len(filenames)}", flush=True)
        resized.flush()
        del resized
        metadata_temp.write_text(json.dumps(expected, ensure_ascii=False) + "\n", encoding="utf-8")
        array_temp.replace(array_path)
        metadata_temp.replace(metadata_path)
        print(f"created cache: {array_path.resolve()}")
    else:
        print(f"reusing cache: {array_path.resolve()}")

    image_array = np.load(array_path, mmap_mode="r")
    return image_array, {name: i for i, name in enumerate(filenames)}


train_image_array, train_image_index = prepare_image_memmap("train", train_inputs)
val_image_array, val_image_index = prepare_image_memmap("validation", val_inputs)
test_image_array, test_image_index = prepare_image_memmap("test", test_inputs)
print("고유 이미지:", len(train_image_index), len(val_image_index), len(test_image_index))

## 3. 태양 원반 검출 · 코로나홀 마스크 피처 추출 — **P3-renew 핵심**

격자 binning 을 걷어내고, **마스크에서 직접** 프레임당 스칼라를 뽑는다.

1. **원반 검출** — `v1` 은 P3 그대로(밝은 픽셀 넓이 -> 반지름, x0.95).
   `v2` 는 림 기울기 최대점에 원을 최소제곱 적합하고 x0.90.
2. **코로나홀 마스크** — `v1` 은 P3 그대로(193 **AND** 211 이 원반 중앙값의
   `CH_THRESHOLD_RATIO` 미만). `v2` 는 반경 프로파일로 평탄화한 뒤 조용한 코로나
   **최빈값** 을 기준으로 삼고, 아주 작은 조각을 버린다.
3. **요약** — 격자 대신 마스크 전체에서:

   | 피처 | 정의 | `area` | `area_shape` |
   |---|---|:--:|:--:|
   | `ch_area` | 유효 원반 대비 코로나홀 픽셀 비율 | O | O |
   | `ch_cx` | 마스크 무게중심의 동서 위치 (원반 반지름으로 정규화, + = 서쪽 림) | | O |
   | `ch_cy` | 무게중심의 남북 위치 | | O |
   | `ch_spread` | 무게중심 기준 rms 거리 (홀이 뭉쳤나 흩어졌나) | | O |
   | `ch_depth` | 마스크 안 193 평균 밝기의 기준 대비 어두움 `1 - I/ref` | | O |

   모멘트는 격자와 달리 **경계가 없다.** 원반 검출이 몇 픽셀 흔들려도 값이
   튀지 않고, 홀이 셀 경계를 넘어갈 때 생기던 계단도 없다.

결과 `(n_images, N_CH_FEATURES)` 는 P3 의 `(n_images, 15)` 자리에 그대로 들어가므로
아래 Dataset · 모델 · 탄도 정렬 셀은 **P3 원본 그대로** 쓴다.

In [ ]:
try:
    from scipy import ndimage as _ndimage
except ImportError:
    _ndimage = None


# ===== 1) 원반 기하 ===================================================
def detect_disk(image_array, sample_count=400):
    # P3 원본과 같은 넓이 역산. v2 에서는 림 적합의 씨앗으로만 쓴다.
    indexes = np.unique(np.linspace(0, len(image_array) - 1, sample_count).astype(int))
    mean_image = np.asarray(image_array[indexes], dtype=np.float64).mean(axis=(0, 1))
    # 배경은 어둡고 원반은 밝습니다. 단순 임계로 원반 픽셀을 잡습니다.
    mask = mean_image > mean_image.max() * 0.15
    ys, xs = np.nonzero(mask)
    center_y, center_x = float(ys.mean()), float(xs.mean())
    radius = float(np.sqrt(mask.sum() / np.pi))
    return center_y, center_x, radius, mean_image


def fit_circle(ys, xs):
    # Kasa 원 적합. x^2+y^2 = 2ax + 2by + c 를 선형 최소제곱으로 풉니다.
    design = np.stack([2 * xs, 2 * ys, np.ones_like(xs)], axis=1)
    target = xs ** 2 + ys ** 2
    (cx, cy, c), *_ = np.linalg.lstsq(design, target, rcond=None)
    return float(cy), float(cx), float(np.sqrt(max(c + cx ** 2 + cy ** 2, 1e-6)))


def fit_limb(mean_image, seed_y, seed_x, seed_r, n_rays=720, iterations=2):
    # 방향별로 밝기 기울기가 가장 급한 반지름 = 림. 그 점들에 원을 적합합니다.
    size = mean_image.shape[0]
    center_y, center_x, radius = seed_y, seed_x, seed_r
    for _ in range(iterations):
        angles = np.linspace(0, 2 * np.pi, n_rays, endpoint=False)
        steps = np.arange(0.55 * radius, 1.35 * radius, 0.25)
        ys = center_y + steps[None, :] * np.sin(angles)[:, None]
        xs = center_x + steps[None, :] * np.cos(angles)[:, None]
        inside = (ys >= 0) & (ys <= size - 1) & (xs >= 0) & (xs <= size - 1)
        samples = mean_image[np.clip(np.rint(ys), 0, size - 1).astype(int),
                             np.clip(np.rint(xs), 0, size - 1).astype(int)]
        samples = np.where(inside, samples, np.nan)
        kernel = np.ones(3) / 3.0
        smooth = np.apply_along_axis(lambda row: np.convolve(row, kernel, "same"), 1, samples)
        gradient = np.diff(smooth, axis=1)
        usable = np.isfinite(gradient).all(axis=1)
        if usable.sum() < n_rays * 0.5:
            gradient = np.nan_to_num(gradient, nan=0.0)
            usable = np.ones(n_rays, dtype=bool)
        index = np.argmin(np.where(np.isfinite(gradient), gradient, 0.0), axis=1)
        depth = -np.take_along_axis(gradient, index[:, None], axis=1).ravel()
        found = steps[index] + 0.5 * (steps[1] - steps[0])
        keep = usable & (depth > np.nanmedian(depth) * 0.25)
        deviation = np.abs(found - np.median(found[keep]))
        keep &= deviation < 4.0 * (np.median(deviation[keep]) + 1e-6)
        center_y, center_x, radius = fit_circle(
            center_y + found[keep] * np.sin(angles[keep]),
            center_x + found[keep] * np.cos(angles[keep]))
    return center_y, center_x, radius, int(keep.sum())


SEED_Y, SEED_X, SEED_R, MEAN_IMAGE = detect_disk(train_image_array)
if CH_MASK_VERSION == "v2":
    DISK_Y, DISK_X, DISK_R, FIT_RAYS = fit_limb(MEAN_IMAGE, SEED_Y, SEED_X, SEED_R)
    print(f"원반 적합(v2): center=({DISK_Y:.1f}, {DISK_X:.1f}) radius={DISK_R:.1f}px "
          f"({FIT_RAYS}/720 방향 채택)")
    print(f"    넓이 역산(v1) 반지름 {SEED_R:.1f}px 대비 {DISK_R / SEED_R - 1:+.1%}")
else:
    DISK_Y, DISK_X, DISK_R = SEED_Y, SEED_X, SEED_R
    print(f"원반 검출(v1): center=({DISK_Y:.1f}, {DISK_X:.1f}) radius={DISK_R:.1f}px")
EFFECTIVE_R = DISK_R * EFFECTIVE_MARGIN

grid_y, grid_x = np.mgrid[0:IMAGE_SIZE, 0:IMAGE_SIZE].astype(np.float64)
RADIUS_MAP = np.sqrt((grid_y - DISK_Y) ** 2 + (grid_x - DISK_X) ** 2)
DISK_MASK = RADIUS_MAP <= EFFECTIVE_R              # 코로나홀을 인정하는 유효 원반
DISK_PIXELS = float(DISK_MASK.sum())
X_NORM = ((grid_x - DISK_X) / DISK_R).astype(np.float32)   # 동서 (+ = 서쪽 림 방향)
Y_NORM = ((grid_y - DISK_Y) / DISK_R).astype(np.float32)   # 남북
X_ON_DISK, Y_ON_DISK = X_NORM[DISK_MASK], Y_NORM[DISK_MASK]
print(f"유효 반지름 {EFFECTIVE_R:.1f}px, 원반 픽셀 {int(DISK_PIXELS):,}개 "
      f"(프레임의 {DISK_PIXELS / IMAGE_SIZE ** 2:.0%})")


# ===== 2) v2 전용 — 반경 프로파일 (train 에서만 만든다) ================
PROFILE_MAP = None
if CH_MASK_VERSION == "v2":
    stack_index = np.unique(np.linspace(0, len(train_image_array) - 1,
                                        V2_STACK_COUNT).astype(int))
    stack = np.asarray(train_image_array[stack_index], dtype=np.float32)
    radial_bin = np.clip((RADIUS_MAP / DISK_R * RADIAL_BINS).astype(int), 0, RADIAL_BINS - 1)
    on_disk_full = RADIUS_MAP <= DISK_R
    per_frame = np.full((len(stack), len(CHANNELS), RADIAL_BINS), np.nan, dtype=np.float32)
    for bin_index in range(RADIAL_BINS):
        selector = on_disk_full & (radial_bin == bin_index)
        if selector.any():
            per_frame[:, :, bin_index] = np.median(stack[:, :, selector], axis=2)
    profile = np.nanmedian(per_frame, axis=0)                      # (2, RADIAL_BINS)
    for channel_index in range(len(CHANNELS)):
        row = profile[channel_index]
        bad = ~np.isfinite(row) | (row <= 0)
        if bad.any():
            row[bad] = np.interp(np.flatnonzero(bad), np.flatnonzero(~bad), row[~bad])
    # 시간중앙값이라 한 프레임의 홀이 자기 기준을 지우지 않는다 (극지방 홀 보호).
    PROFILE_MAP = profile[:, radial_bin].astype(np.float32)        # (2, H, W)
    del stack, per_frame
    print(f"반경 프로파일: {len(stack_index)}프레임 x {RADIAL_BINS}구간 -> 평탄화 준비 완료")
    if V2_MIN_AREA > 0 and _ndimage is None:
        print("  주의: scipy 가 없어 작은 조각 제거를 건너뜁니다 (V2_MIN_AREA 무시)")


# ===== 3) 코로나홀 마스크 =============================================
def quiet_level(values, bins=256, span=(0.0, 3.0)):
    # 평탄화한 원반 밝기의 히스토그램 봉우리 = 조용한 코로나 수준.
    # 중앙값과 달리 활동영역이 원반의 몇 %를 덮든 거의 움직이지 않는다.
    histogram, edges = np.histogram(values, bins=bins, range=span)
    smooth = np.convolve(histogram.astype(np.float64), np.ones(9) / 9.0, "same")
    centers = 0.5 * (edges[:-1] + edges[1:])
    return float(centers[int(np.argmax(smooth))])


def drop_small_components(mask):
    if _ndimage is None or V2_MIN_AREA <= 0:
        return mask
    limit = V2_MIN_AREA * DISK_PIXELS
    structure = np.ones((3, 3), dtype=bool)
    result = np.zeros_like(mask)
    for index in range(len(mask)):
        if not mask[index].any():
            continue
        labels, count = _ndimage.label(mask[index], structure=structure)
        if not count:
            continue
        sizes = np.bincount(labels.ravel())
        keep = sizes >= limit
        keep[0] = False
        result[index] = keep[labels]
    return result


def coronal_hole_masks(block):
    # block: (n, C, H, W) float32. 반환: 마스크 (n, H, W), 기준값 (n, C), 비교에 쓴 밝기
    if CH_MASK_VERSION == "v2":
        values = block / np.maximum(PROFILE_MAP, 1e-3)
        reference = np.empty((len(block), len(CHANNELS)), dtype=np.float32)
        for index in range(len(block)):
            for channel_index in range(len(CHANNELS)):
                reference[index, channel_index] = quiet_level(
                    values[index, channel_index][DISK_MASK])
    else:
        values = block
        reference = np.median(block[:, :, DISK_MASK], axis=2).astype(np.float32)  # (n, C)
    dark = values < (CH_RATIO * reference)[:, :, None, None]
    # 193 과 211 두 채널 모두에서 어두운 픽셀만 코로나홀로 인정 (P3 와 같은 규칙)
    mask = np.logical_and(dark[:, 0], dark[:, 1]) & DISK_MASK
    if CH_MASK_VERSION == "v2":
        mask = drop_small_components(mask)
    return mask, reference, values


# ===== 4) 마스크 -> 피처 ==============================================
def mask_features(mask, reference, values):
    on_disk = mask[:, DISK_MASK].astype(np.float32)                # (n, npix)
    pixel_count = on_disk.sum(axis=1)
    area = pixel_count / DISK_PIXELS
    if CH_FEATURE_MODE == "area":
        return area[:, None].astype(np.float32)

    safe_count = np.maximum(pixel_count, 1.0)
    center_x = on_disk @ X_ON_DISK / safe_count
    center_y = on_disk @ Y_ON_DISK / safe_count
    second_moment = on_disk @ (X_ON_DISK ** 2 + Y_ON_DISK ** 2) / safe_count
    spread = np.sqrt(np.maximum(second_moment - center_x ** 2 - center_y ** 2, 0.0))
    brightness = values[:, 0][:, DISK_MASK]                        # 193 채널
    depth = 1.0 - ((on_disk * brightness).sum(axis=1) / safe_count
                   / np.maximum(reference[:, 0], 1e-6))
    empty = pixel_count <= 0
    for feature in (center_x, center_y, spread, depth):
        feature[empty] = 0.0
    return np.stack([area, center_x, center_y, spread, depth], axis=1).astype(np.float32)


def compute_ch_features(image_array, chunk=256):
    result = np.zeros((len(image_array), N_CH_FEATURES), dtype=np.float32)
    for start in range(0, len(image_array), chunk):
        block = np.asarray(image_array[start:start + chunk], dtype=np.float32)
        mask, reference, values = coronal_hole_masks(block)
        result[start:start + chunk] = mask_features(mask, reference, values)
        if (start // chunk) % 20 == 0 or start + chunk >= len(image_array):
            print(f"  CH {min(start + chunk, len(image_array))}/{len(image_array)}", flush=True)
    return result


def cached_ch_features(split, image_array):
    # 파일명에 마스크 버전·피처 모드·임계비·여유반지름이 들어가므로 P3 캐시와 섞이지 않는다.
    path = CACHE_ROOT / f"chmask_{split}_{CH_CACHE_TAG}.npy"
    if path.exists():
        features = np.load(path)
        if features.shape == (len(image_array), N_CH_FEATURES):
            print(f"reusing CH cache: {path.name}")
            return features
    print(f"extracting CH features: {split}")
    features = compute_ch_features(image_array)
    np.save(path, features)
    print(f"created CH cache: {path.name}  shape={features.shape}")
    return features


train_ch = cached_ch_features("train", train_image_array)
val_ch = cached_ch_features("validation", val_image_array)
test_ch = cached_ch_features("test", test_image_array)
assert train_ch.shape[1] == N_CELLS

print()
print(pd.DataFrame(train_ch, columns=CH_FEATURE_NAMES).describe()
      .loc[["mean", "std", "min", "max"]].round(4))
legacy_path = CACHE_ROOT / f"ch_train_3x5_{CH_THRESHOLD_RATIO}_{IMAGE_SIZE}.npy"
if legacy_path.exists():
    legacy = np.load(legacy_path)
    if len(legacy) == len(train_ch):
        # 셀 평균은 셀 크기가 달라 면적과 정확히 같지 않다. 방향 확인용 참고값이다.
        correlation = np.corrcoef(legacy.mean(axis=1), train_ch[:, 0])[0, 1]
        print(f"\nP3 격자(3x5) 셀평균 vs renew 면적 상관: r={correlation:+.3f}")

figure, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(MEAN_IMAGE, cmap="gray")
axes[0].add_patch(plt.Circle((DISK_X, DISK_Y), EFFECTIVE_R, fill=False,
                             color="red", linewidth=1.5))
if CH_MASK_VERSION == "v2":
    axes[0].add_patch(plt.Circle((SEED_X, SEED_Y), SEED_R * DISK_MARGIN, fill=False,
                                 color="cyan", linewidth=1.0, linestyle="--"))
    axes[0].set_title("train mean + disk (red = v2, cyan = v1)")
else:
    axes[0].set_title("train mean + detected disk")
sample_mask, _, _ = coronal_hole_masks(np.asarray(train_image_array[:1], dtype=np.float32))
axes[1].imshow(sample_mask[0], cmap="gray")
axes[1].set_title(f"CH mask ({CH_MASK_VERSION}) area={sample_mask[0].sum() / DISK_PIXELS:.2%}")
axes[2].plot(train_ch[:800, 0] * 100, linewidth=0.8)
axes[2].set_title("CH area [%] (first 800 frames)")
axes[2].set_xlabel("frame"); axes[2].grid(alpha=0.3)
for axis in axes[:2]:
    axis.set_xticks([]); axis.set_yticks([])
plt.tight_layout(); plt.show()

## 4. 정규화 통계 · 탄도 정렬 인덱스

**탄도 정렬**: horizon $h$ 의 타깃이 발원한 시각은 $T_0 + h - \tau$, $\tau = \mathrm{1AU}/v$.
가정 속도 $v \in \{350, 500, 700\}$ km/s 각각에 대해 윈도우 내 (실수) 인덱스를 계산해 둡니다.
윈도우를 벗어나면 양 끝으로 clamp 합니다 (고속풍 + 장기 horizon 은 아직 관측되지 않은 영역에서 발원).

In [ ]:
def compute_image_stats(array, chunk=256):
    total = np.zeros(len(CHANNELS), np.float64)
    total_square = np.zeros(len(CHANNELS), np.float64)
    count = 0
    for start in range(0, len(array), chunk):
        block = np.asarray(array[start:start + chunk], dtype=np.float64) / 255.0
        total += block.sum(axis=(0, 2, 3))
        total_square += (block ** 2).sum(axis=(0, 2, 3))
        count += block.shape[0] * block.shape[2] * block.shape[3]
    mean = total / count
    return mean.astype(np.float32), np.sqrt(
        np.maximum(total_square / count - mean ** 2, 1e-12)).astype(np.float32)


IMAGE_MEAN, IMAGE_STD = compute_image_stats(train_image_array)
WIND_MEAN = float(train_wind.mean())
WIND_STD = float(train_wind.std() + 1e-6)
DIFF_STD = float(np.diff(train_wind, axis=1, prepend=train_wind[:, :1]).std() + 1e-6)
CH_MEAN = train_ch.mean(axis=0).astype(np.float32)
CH_STD = (train_ch.std(axis=0) + 1e-8).astype(np.float32)

train_residual = train_targets - train_wind[:, -1:]
RESIDUAL_MEAN = train_residual.mean(axis=0).astype(np.float32)
RESIDUAL_STD = (train_residual.std(axis=0) + 1e-6).astype(np.float32)
CLIP_LOW = float(train_targets.min() * 0.95)
CLIP_HIGH = float(train_targets.max() * 1.05)


def ballistic_indices():
    # (12, n_speeds) 윈도우 내 실수 인덱스. 인덱스 19 가 마지막 관측 시점 T0.
    table = np.zeros((12, len(TRANSIT_SPEEDS)), dtype=np.float32)
    for horizon_index in range(12):
        lead_hours = (horizon_index + 1) * 6.0
        for speed_index, speed in enumerate(TRANSIT_SPEEDS):
            transit_hours = AU_KM / speed / 3600.0
            table[horizon_index, speed_index] = np.clip(
                19.0 + (lead_hours - transit_hours) / 6.0, 0.0, 19.0)
    return table


BALLISTIC_INDEX = ballistic_indices()
N_SPEEDS = len(TRANSIT_SPEEDS)


def image_index_matrix(inputs, image_index):
    return np.asarray([
        [image_index[name] for name in row]
        for row in inputs[IMAGE_COLUMNS].itertuples(index=False, name=None)
    ], dtype=np.int32)


def compute_ballistic(ch_grid, indexes):
    # 중앙자오선 셀 시계열을 탄도 역산 인덱스에서 선형보간 -> (n, 12, n_speeds)
    central = ch_grid[indexes][:, :, CENTRAL_CELL]
    lower = np.floor(BALLISTIC_INDEX).astype(np.int64)
    upper = np.minimum(lower + 1, 19)
    weight = (BALLISTIC_INDEX - lower).astype(np.float32)
    return (central[:, lower] * (1.0 - weight) + central[:, upper] * weight).astype(np.float32)


# 탄도 피처 정규화 통계도 반드시 train split 에서만 산출합니다.
_train_ballistic = compute_ballistic(train_ch, image_index_matrix(train_inputs, train_image_index))
BALLISTIC_MEAN = float(_train_ballistic.mean())
BALLISTIC_STD = float(_train_ballistic.std() + 1e-8)

print("탄도 역산 — 가정 속도별 전달 시간:")
for speed in TRANSIT_SPEEDS:
    print(f"  v={speed:5.0f} km/s -> tau = {AU_KM / speed / 3600.0:5.1f} h "
          f"({AU_KM / speed / 86400.0:.2f} 일)")
print("\nhorizon별 근원 시점 인덱스 (19 = 마지막 관측):")
print(pd.DataFrame(BALLISTIC_INDEX, index=[f"{h}h" for h in HORIZONS],
                   columns=[f"v={int(v)}" for v in TRANSIT_SPEEDS]).round(2))
print(f"\nresidual std by horizon: {np.round(RESIDUAL_STD, 1)}")
print(f"clip range: [{CLIP_LOW:.1f}, {CLIP_HIGH:.1f}] km/s")

## 5. Dataset

In [ ]:
STAT_NAMES = ["last", "mean4", "mean", "std", "min", "max", "slope", "last_minus_mean4", "range"]
_TIME_CENTERED = np.arange(20, dtype=np.float32) - 9.5
_TIME_DENOMINATOR = float((_TIME_CENTERED ** 2).sum())


def build_wind_stats(wind):
    last = wind[:, -1]
    mean4 = wind[:, -4:].mean(axis=1)
    slope = (wind - wind.mean(axis=1, keepdims=True)) @ _TIME_CENTERED / _TIME_DENOMINATOR
    return np.stack([last, mean4, wind.mean(axis=1), wind.std(axis=1), wind.min(axis=1),
                     wind.max(axis=1), slope, last - mean4,
                     wind.max(axis=1) - wind.min(axis=1)], axis=1).astype(np.float32)


train_stats_raw = build_wind_stats(train_wind)
STATS_MEAN = train_stats_raw.mean(axis=0).astype(np.float32)
STATS_STD = (train_stats_raw.std(axis=0) + 1e-6).astype(np.float32)
NUM_STATS = len(STAT_NAMES)


class SolarWindDataset(Dataset):
    def __init__(self, image_array, image_index, inputs, wind, wind_valid,
                 ch_grid, targets=None, training=False):
        self.training = training
        self.image_array = image_array
        self.image_indexes = image_index_matrix(inputs, image_index)
        self.sample_ids = inputs.sample_id.to_numpy()
        self.last_wind = np.ascontiguousarray(wind[:, -1]).astype(np.float32)
        self.wind_seq = np.stack([
            (wind - WIND_MEAN) / WIND_STD,
            np.diff(wind, axis=1, prepend=wind[:, :1]) / DIFF_STD,
            wind_valid,
        ], axis=2).astype(np.float32)
        self.wind_stats = ((build_wind_stats(wind) - STATS_MEAN) / STATS_STD).astype(np.float32)

        # (n_samples, 20, N_CELLS) 시퀀스로 미리 펼쳐 둡니다.
        self.ch_seq = ((ch_grid[self.image_indexes] - CH_MEAN) / CH_STD).astype(np.float32)
        # 탄도 정렬 피처. 정규화는 train 통계(BALLISTIC_MEAN/STD)로 고정합니다.
        self.ballistic = ((compute_ballistic(ch_grid, self.image_indexes) - BALLISTIC_MEAN)
                          / BALLISTIC_STD).astype(np.float32)           # (n, 12, n_speeds)

        self.targets = targets.astype(np.float32) if targets is not None else None
        self.image_mean = IMAGE_MEAN.reshape(1, len(CHANNELS), 1, 1)
        self.image_std = IMAGE_STD.reshape(1, len(CHANNELS), 1, 1)

    def __len__(self):
        return len(self.sample_ids)

    def __getitem__(self, item):
        if USE_CNN:
            images = np.asarray(
                self.image_array[self.image_indexes[item]], dtype=np.float32) / 255.0
            images = ((images - self.image_mean) / self.image_std).astype(np.float32)
        else:
            images = np.zeros((1, 1, 1, 1), dtype=np.float32)

        ch_seq = self.ch_seq[item]
        ballistic = self.ballistic[item]
        if self.training and AUGMENT and AUG_CH_NOISE > 0:
            # CH 피처에도 약한 곱셈 노이즈를 줘 브랜치 과적합을 억제합니다.
            ch_seq = ch_seq * (1.0 + np.random.normal(0, AUG_CH_NOISE, ch_seq.shape)
                               ).astype(np.float32)
            ballistic = ballistic * (1.0 + np.random.normal(
                0, AUG_CH_NOISE, ballistic.shape)).astype(np.float32)

        result = {
            "images": torch.from_numpy(np.ascontiguousarray(images)),
            "wind_seq": torch.from_numpy(self.wind_seq[item]),
            "wind_stats": torch.from_numpy(self.wind_stats[item]),
            "ch_seq": torch.from_numpy(np.ascontiguousarray(ch_seq)),
            "ballistic": torch.from_numpy(np.ascontiguousarray(ballistic)),
            "last_wind": torch.tensor(self.last_wind[item]),
            "sample_id": self.sample_ids[item],
        }
        if self.targets is not None:
            result["target"] = torch.from_numpy(self.targets[item])
        return result


def seed_worker(worker_id):
    worker_seed = (SEED + worker_id) % (2 ** 32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)


def make_loader(dataset, shuffle):
    options = dict(dataset=dataset, batch_size=BATCH_SIZE, shuffle=shuffle,
                   num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, drop_last=False,
                   worker_init_fn=seed_worker,
                   generator=torch.Generator().manual_seed(SEED))
    if NUM_WORKERS > 0:
        options.update(persistent_workers=True, prefetch_factor=2)
    return DataLoader(**options)


train_dataset = SolarWindDataset(train_image_array, train_image_index, train_inputs,
                                 train_wind, train_wind_valid, train_ch,
                                 train_targets, training=True)
val_dataset = SolarWindDataset(val_image_array, val_image_index, val_inputs,
                               val_wind, val_wind_valid, val_ch, val_targets)
train_loader = make_loader(train_dataset, shuffle=True)
val_loader = make_loader(val_dataset, shuffle=False)

batch = next(iter(train_loader))
assert batch["ch_seq"].shape[1:] == (20, N_CELLS)
assert batch["ballistic"].shape[1:] == (12, N_SPEEDS)
print({k: tuple(v.shape) for k, v in batch.items() if torch.is_tensor(v)})

## 6. 모델

- **CH 브랜치** — 마스크 피처 시퀀스 `(20, N_CELLS)` → GRU.
  `N_CELLS` 는 격자 셀이 아니라 **CH 피처 개수**다 (기본 1 = 면적).
  P3(15) 보다 입력이 좁아진 것 말고는 구조가 같다
- **Wind 브랜치** — P1 과 동일 (GRU + 통계)
- **CNN 브랜치** — `USE_CNN=True` 일 때만. 기본은 꺼짐
- **head** — horizon 12개에 **가중치를 공유**하고, horizon embedding 과
  해당 horizon 의 탄도 피처만 다르게 넣습니다. 파라미터가 12배 줄어 정규화 효과가 큽니다

In [ ]:
class Inception3D(nn.Module):
    def __init__(self, in_channels, branch_channels=32):
        super().__init__()
        self.branch_1 = nn.Sequential(
            nn.Conv3d(in_channels, branch_channels, 1), nn.ReLU(inplace=True))
        self.branch_3 = nn.Sequential(
            nn.Conv3d(in_channels, branch_channels, 1), nn.ReLU(inplace=True),
            nn.Conv3d(branch_channels, branch_channels, (1, 3, 3), padding=(0, 1, 1)),
            nn.ReLU(inplace=True))
        self.branch_5 = nn.Sequential(
            nn.Conv3d(in_channels, branch_channels, 1), nn.ReLU(inplace=True),
            nn.Conv3d(branch_channels, branch_channels, (1, 5, 5), padding=(0, 2, 2)),
            nn.ReLU(inplace=True))
        self.branch_pool = nn.Sequential(
            nn.MaxPool3d((1, 3, 3), stride=1, padding=(0, 1, 1)),
            nn.Conv3d(in_channels, branch_channels, 1), nn.ReLU(inplace=True))

    def forward(self, x):
        return torch.cat([self.branch_1(x), self.branch_3(x),
                          self.branch_5(x), self.branch_pool(x)], dim=1)


class SolarWindP3(nn.Module):
    def __init__(self):
        super().__init__()
        shared_dim = 0

        self.wind_gru = nn.GRU(3, 96, num_layers=2, batch_first=True)
        self.stats_encoder = nn.Sequential(
            nn.Linear(NUM_STATS, 128), nn.SELU(inplace=True),
            nn.Linear(128, 64), nn.SELU(inplace=True))
        shared_dim += 96 + 64

        if USE_CH:
            self.ch_gru = nn.GRU(N_CELLS, 64, num_layers=2, batch_first=True)
            self.ch_dropout = nn.Dropout(DROPOUT)
            shared_dim += 64

        if USE_CNN:
            self.stem = nn.Sequential(
                nn.Conv3d(len(CHANNELS), 32, (1, 5, 5), padding=(0, 2, 2)),
                nn.BatchNorm3d(32), nn.ReLU(inplace=True),
                nn.MaxPool3d((1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1)),
                nn.Conv3d(32, 64, (1, 3, 3), padding=(0, 1, 1)),
                nn.BatchNorm3d(64), nn.ReLU(inplace=True),
                nn.MaxPool3d((1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1)))
            blocks, in_channels = [], 64
            for _ in range(3):
                blocks.extend([Inception3D(in_channels, 32),
                               nn.MaxPool3d((1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1))])
                in_channels = 128
            self.image_encoder = nn.Sequential(*blocks)
            self.image_lstm = nn.LSTM(128 * 1 * 4, 128, batch_first=True)
            self.image_dropout = nn.Dropout(DROPOUT)
            shared_dim += 128

        self.horizon_embedding = nn.Parameter(torch.randn(12, HORIZON_EMBED) * 0.1)
        head_input = shared_dim + HORIZON_EMBED + (N_SPEEDS if USE_BALLISTIC else 0)
        # head 는 12 horizon 에 동일 가중치로 적용됩니다 (nn.Linear 는 마지막 축에만 작용).
        self.head = nn.Sequential(
            nn.Linear(head_input, 192), nn.ReLU(inplace=True), nn.Dropout(DROPOUT),
            nn.Linear(192, 96), nn.ReLU(inplace=True), nn.Dropout(DROPOUT),
            nn.Linear(96, 1))

        self.register_buffer("residual_mean", torch.as_tensor(RESIDUAL_MEAN))
        self.register_buffer("residual_std", torch.as_tensor(RESIDUAL_STD))
        print(f"shared_dim={shared_dim}  head_input={head_input}")

    def forward(self, images, wind_seq, wind_stats, ch_seq, ballistic):
        _, wind_hidden = self.wind_gru(wind_seq)
        parts = [F.relu(wind_hidden[-1]), self.stats_encoder(wind_stats)]

        if USE_CH:
            _, ch_hidden = self.ch_gru(ch_seq)
            parts.append(self.ch_dropout(F.relu(ch_hidden[-1])))

        if USE_CNN:
            features = images.permute(0, 2, 1, 3, 4).contiguous()
            features = self.image_encoder(self.stem(features))
            features = F.adaptive_avg_pool3d(features, (features.shape[2], 1, 4))
            features = features.permute(0, 2, 1, 3, 4).flatten(2)
            _, (hidden, _) = self.image_lstm(features)
            parts.append(self.image_dropout(F.relu(hidden[-1])))

        shared = torch.cat(parts, dim=1)                                   # (B, D)
        batch_size = shared.shape[0]
        expanded = shared.unsqueeze(1).expand(batch_size, 12, shared.shape[1])
        embedding = self.horizon_embedding.unsqueeze(0).expand(batch_size, 12, HORIZON_EMBED)
        head_parts = [expanded, embedding]
        if USE_BALLISTIC:
            head_parts.append(ballistic)
        z = self.head(torch.cat(head_parts, dim=2)).squeeze(-1)            # (B, 12)
        return z * self.residual_std + self.residual_mean


def build_model():
    return SolarWindP3().to(DEVICE)


model = build_model()
print("trainable parameters:",
      f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 7. 지표 · 손실

In [ ]:
def official_rmse(y_true, y_pred):
    per_horizon = np.sqrt(np.mean((y_pred - y_true) ** 2, axis=0))
    return float(per_horizon.mean()), per_horizon


def pooled_rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_pred - y_true) ** 2)))


def metrics_by_horizon(y_true, y_pred, persistence=None):
    rows = []
    for index in range(12):
        actual, predicted = y_true[:, index], y_pred[:, index]
        error = predicted - actual
        denominator = np.std(actual) * np.std(predicted)
        row = {"horizon_h": int(HORIZONS[index]),
               "rmse": float(np.sqrt(np.mean(error ** 2))),
               "mae": float(np.mean(np.abs(error))),
               "corr": float(np.corrcoef(actual, predicted)[0, 1]) if denominator > 0 else np.nan}
        if persistence is not None:
            row["persistence_rmse"] = float(np.sqrt(np.mean((persistence[:, index] - actual) ** 2)))
            row["gain"] = row["persistence_rmse"] - row["rmse"]
        rows.append(row)
    return pd.DataFrame(rows)


def metric_loss(prediction, target):
    error = (prediction - target) / LOSS_SCALE
    if LOSS_MODE == "mse":
        return (error ** 2).mean()
    return torch.sqrt((error ** 2).mean(dim=0) + LOSS_EPSILON).mean()


def augment_batch(images):
    # GPU 에서 수행합니다. numpy 증강은 CPU 병목으로 epoch 시간이 4배 늘었습니다.
    if AUG_SHIFT_PIXELS > 0:
        shift_y = int(torch.randint(-AUG_SHIFT_PIXELS, AUG_SHIFT_PIXELS + 1, (1,)).item())
        shift_x = int(torch.randint(-AUG_SHIFT_PIXELS, AUG_SHIFT_PIXELS + 1, (1,)).item())
        images = torch.roll(images, shifts=(shift_y, shift_x), dims=(3, 4))
    count = images.shape[0]
    scale = 1.0 + (torch.rand(count, 1, 1, 1, 1, device=images.device) * 2 - 1) * AUG_BRIGHTNESS
    offset = torch.randn(count, 1, 1, 1, 1, device=images.device) * AUG_BRIGHTNESS
    images = images * scale + offset
    if AUG_NOISE_STD > 0:
        images = images + torch.randn_like(images) * AUG_NOISE_STD
    if AUG_ERASE_PROB > 0:
        size = max(4, IMAGE_SIZE // 8)
        selected = torch.rand(count, device=images.device) < AUG_ERASE_PROB
        if bool(selected.any()):
            top = int(torch.randint(0, IMAGE_SIZE - size, (1,)).item())
            left = int(torch.randint(0, IMAGE_SIZE - size, (1,)).item())
            images[selected, :, :, top:top + size, left:left + size] = 0.0
    return images


val_persistence = np.repeat(val_wind[:, -1:], 12, axis=1).astype(np.float64)
persistence_score, persistence_per_horizon = official_rmse(val_targets, val_persistence)
print(f"[기준선] persistence 공식 RMSE = {persistence_score:.3f} km/s")
print("horizon별:", np.round(persistence_per_horizon, 1))

## 8. 학습

In [ ]:
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

model = build_model()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=SCHEDULER_PATIENCE, min_lr=1e-6)
scaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_AMP)

checkpoint_path = OUTPUT_DIR / "best_model.pth"
if checkpoint_path.exists():
    checkpoint_path.unlink()
best_val_score = float("inf")
epochs_without_improvement = 0
history = []

CONFIG = {"image_size": IMAGE_SIZE, "channels": list(CHANNELS), "use_cnn": USE_CNN,
          "use_ch": USE_CH, "use_ballistic": USE_BALLISTIC, "ch_grid": list(CH_GRID),
          "ch_threshold_ratio": CH_RATIO, "disk": [DISK_Y, DISK_X, DISK_R],
          "ch_mask_version": CH_MASK_VERSION, "ch_feature_mode": CH_FEATURE_MODE,
          "ch_features": CH_FEATURE_NAMES, "effective_margin": EFFECTIVE_MARGIN,
          "transit_speeds": list(TRANSIT_SPEEDS), "seed": SEED,
          "image_mean": IMAGE_MEAN.tolist(), "image_std": IMAGE_STD.tolist(),
          "wind_mean": WIND_MEAN, "wind_std": WIND_STD, "diff_std": DIFF_STD,
          "ch_mean": CH_MEAN.tolist(), "ch_std": CH_STD.tolist(),
          "stats_mean": STATS_MEAN.tolist(), "stats_std": STATS_STD.tolist(),
          "residual_mean": RESIDUAL_MEAN.tolist(), "residual_std": RESIDUAL_STD.tolist(),
          "clip_low": CLIP_LOW, "clip_high": CLIP_HIGH,
          "initialization": "random_from_scratch"}


def run_epoch(loader, training):
    model.train(training)
    squared_error_sum = np.zeros(12, dtype=np.float64)
    sample_count = 0
    for batch in loader:
        images = batch["images"].to(DEVICE, non_blocking=PIN_MEMORY)
        wind_seq = batch["wind_seq"].to(DEVICE, non_blocking=PIN_MEMORY)
        wind_stats = batch["wind_stats"].to(DEVICE, non_blocking=PIN_MEMORY)
        ch_seq = batch["ch_seq"].to(DEVICE, non_blocking=PIN_MEMORY)
        ballistic = batch["ballistic"].to(DEVICE, non_blocking=PIN_MEMORY)
        last_wind = batch["last_wind"].to(DEVICE, non_blocking=PIN_MEMORY)
        target = batch["target"].to(DEVICE, non_blocking=PIN_MEMORY)

        if training and USE_CNN and AUGMENT:
            images = augment_batch(images)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            with torch.amp.autocast(DEVICE.type, enabled=USE_AMP):
                residual = model(images, wind_seq, wind_stats, ch_seq, ballistic)
            prediction = residual.float() + last_wind.unsqueeze(1)
            loss = metric_loss(prediction, target)
            if training:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()

        error = (prediction.detach() - target).double()
        squared_error_sum += torch.sum(error ** 2, dim=0).cpu().numpy()
        sample_count += error.shape[0]
    per_horizon = np.sqrt(squared_error_sum / sample_count)
    return float(per_horizon.mean()), per_horizon


for epoch in range(1, EPOCHS + 1):
    started = time.perf_counter()
    train_score, _ = run_epoch(train_loader, training=True)
    with torch.no_grad():
        val_score, _ = run_epoch(val_loader, training=False)
    scheduler.step(val_score)
    learning_rate = optimizer.param_groups[0]["lr"]
    elapsed = time.perf_counter() - started
    history.append({"epoch": epoch, "train_rmse": train_score, "val_rmse": val_score,
                    "learning_rate": learning_rate, "seconds": elapsed})
    marker = ""
    if val_score < best_val_score:
        best_val_score = val_score
        epochs_without_improvement = 0
        torch.save({"model_state_dict": model.state_dict(), "epoch": epoch,
                    "val_official_rmse": val_score, **CONFIG}, checkpoint_path)
        marker = "  <- best"
    else:
        epochs_without_improvement += 1
    print(f"epoch={epoch:03d} train={train_score:7.3f} val={val_score:7.3f} "
          f"lr={learning_rate:.2e} {elapsed:6.1f}s{marker}", flush=True)
    if epochs_without_improvement >= EARLY_STOP_PATIENCE:
        print("early stopping")
        break

history_frame = pd.DataFrame(history)
history_frame.to_csv(OUTPUT_DIR / "history.csv", index=False)
figure, axis = plt.subplots(figsize=(7, 4))
axis.plot(history_frame.epoch, history_frame.train_rmse, label="train")
axis.plot(history_frame.epoch, history_frame.val_rmse, label="validation")
axis.axhline(persistence_score, color="gray", linestyle="--", label="persistence")
axis.set_xlabel("epoch"); axis.set_ylabel("official RMSE (km/s)")
axis.grid(alpha=0.3); axis.legend()
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "learning_curve.png", dpi=140); plt.show()

checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=True)
model.load_state_dict(checkpoint["model_state_dict"])
print(f"best epoch {checkpoint['epoch']}  val official RMSE {checkpoint['val_official_rmse']:.3f}")

## 9. Validation 평가

In [ ]:
@torch.no_grad()
def predict(loader):
    model.eval()
    predictions, sample_ids = [], []
    for batch in loader:
        images = batch["images"].to(DEVICE, non_blocking=PIN_MEMORY)
        wind_seq = batch["wind_seq"].to(DEVICE, non_blocking=PIN_MEMORY)
        wind_stats = batch["wind_stats"].to(DEVICE, non_blocking=PIN_MEMORY)
        ch_seq = batch["ch_seq"].to(DEVICE, non_blocking=PIN_MEMORY)
        ballistic = batch["ballistic"].to(DEVICE, non_blocking=PIN_MEMORY)
        last_wind = batch["last_wind"].to(DEVICE, non_blocking=PIN_MEMORY)
        with torch.amp.autocast(DEVICE.type, enabled=USE_AMP):
            residual = model(images, wind_seq, wind_stats, ch_seq, ballistic)
        prediction = (residual.float() + last_wind.unsqueeze(1)).clamp(CLIP_LOW, CLIP_HIGH)
        predictions.append(prediction.cpu().numpy())
        sample_ids.extend(batch["sample_id"])
    return np.concatenate(predictions).astype(np.float64), sample_ids


validation_prediction, validation_ids = predict(val_loader)
assert validation_ids == val_inputs.sample_id.tolist()
model_score, _ = official_rmse(val_targets, validation_prediction)
validation_metrics = metrics_by_horizon(val_targets, validation_prediction, val_persistence)
validation_metrics.to_csv(OUTPUT_DIR / "validation_metrics.csv", index=False)

print(f"공식 RMSE (mean of horizon RMSE) : {model_score:8.3f} km/s")
print(f"pooled RMSE (전체 원소)          : {pooled_rmse(val_targets, validation_prediction):8.3f} km/s")
print(f"persistence 공식 RMSE            : {persistence_score:8.3f} km/s")
print(f"persistence 대비 개선             : {persistence_score - model_score:8.3f} km/s"
      f"  ({(persistence_score - model_score) / persistence_score:.1%})")
print("\n[참고] P1 = 68.408 / P2a = 65.663 / P3(3x5 격자) = 64.203")
if model_score >= persistence_score:
    print("\n>>> 경고: persistence 미달. 제출하지 마세요.")

figure, axis = plt.subplots(figsize=(7, 4))
axis.plot(validation_metrics.horizon_h, validation_metrics.rmse, marker="o", label="model")
axis.plot(validation_metrics.horizon_h, validation_metrics.persistence_rmse,
          marker="s", linestyle="--", label="persistence")
axis.set_xlabel("forecast horizon (h)"); axis.set_ylabel("RMSE (km/s)")
axis.grid(alpha=0.3); axis.legend()
plt.tight_layout(); plt.show()
validation_metrics

## 10. Test 추론 · 제출 파일 생성

In [ ]:
del train_loader, val_loader, train_dataset, val_dataset
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

test_dataset = SolarWindDataset(test_image_array, test_image_index, test_inputs,
                                test_wind, test_wind_valid, test_ch, targets=None)
test_loader = make_loader(test_dataset, shuffle=False)
test_prediction, predicted_ids = predict(test_loader)

assert predicted_ids == test_inputs.sample_id.tolist()
assert test_prediction.shape == (len(test_inputs), 12)
assert np.isfinite(test_prediction).all()

submission = pd.DataFrame(test_prediction, columns=TARGET_COLUMNS)
submission.insert(0, "sample_id", predicted_ids)
submission.to_csv(SUBMISSION_DIR / "submission.csv", index=False)
shutil.copyfile(checkpoint_path, SUBMISSION_DIR / "model.pth")

# 규정: "code.ipynb 에서 model.pth 를 불러와 추론이 가능해야 함" 을 문자 그대로 충족
saved = torch.load(SUBMISSION_DIR / "model.pth", map_location=DEVICE, weights_only=True)
model.load_state_dict(saved["model_state_dict"])
print("reloaded from submission/model.pth")

print("saved:", (SUBMISSION_DIR / "submission.csv").resolve(), submission.shape)
print(submission[TARGET_COLUMNS].describe().loc[["mean", "std", "min", "max"]].round(1))
del test_dataset, test_loader
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
submission.head()

## 11. 제출 전 체크리스트

> ⚠️ `code.ipynb` 는 **수동 복사**입니다. 저장(Ctrl+S) 후 `submission/code.ipynb` 로 복사하세요.

In [ ]:
EXPECTED_TEST_ROWS = 3868

print("=== 제출 점검 ===")
ok = True
for name in ["code.ipynb", "model.pth", "submission.csv"]:
    path = SUBMISSION_DIR / name
    if path.exists():
        print(f"  [O] {name:16s} {path.stat().st_size / 1024 ** 2:8.2f} MiB")
    else:
        print(f"  [X] {name:16s} 없음")
        ok = False

check = pd.read_csv(SUBMISSION_DIR / "submission.csv")
print(f"\n  행 수      : {len(check):,} (기대 {EXPECTED_TEST_ROWS:,})",
      "OK" if len(check) == EXPECTED_TEST_ROWS else "<-- 불일치")
print(f"  컬럼       : {check.columns.tolist() == ['sample_id'] + TARGET_COLUMNS}")
print(f"  결측       : {int(check[TARGET_COLUMNS].isna().sum().sum())}")
print(f"  sample_id  : 유일={check.sample_id.is_unique}, "
      f"test_ids 일치={sorted(check.sample_id) == sorted(test_ids.sample_id)}")
print(f"  값 범위    : [{check[TARGET_COLUMNS].to_numpy().min():.1f}, "
      f"{check[TARGET_COLUMNS].to_numpy().max():.1f}] km/s")

extra = [p.name for p in SUBMISSION_DIR.iterdir()
         if p.name not in {"code.ipynb", "model.pth", "submission.csv"}]
if extra:
    print(f"\n  주의: 불필요한 파일 -> {extra}")